# T382 — RAL Silver ARA-native detector-share test

## TL;DR

**Frozen verdict:** `PARENT_RECOVERED_96_DETECTOR_CHILD_NOT_QUALIFIED`.

The detector-summed population parent passed at τ = 2.192800 μs. The
calibration-frozen 96-detector child did not pass its validation/holdout and
detector-map controls, so C06 is not interpreted as child-mediated decay
timing. C16 is unavailable because the archive contains aggregate histograms,
not individually linked muons and daughters.

This notebook is the reviewable companion to the executed Python module. It
keeps ARA parent, native child, projected child, and established-physics
crosswalks separate.


## Who / what / when / where / why / how

- **Who/where:** untouched RAL Silver runs from ISIS EMU study 10.5286/ISIS.E.RB1620201.
- **What:** population parent and a calibration-frozen 96-detector traversal child.
- **When:** native 0.016 μs bins in the frozen 0.25–8.00 μs analysis window.
- **Why:** test child-native `0→2→0`, projection to the parent ridge, and advance information without relabelling detector amplitudes as individual muons.
- **How:** calibration/validation/holdout field ladder, no-phase/reverse/detector-shift controls, bootstrap and bin sensitivity.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import sys

HERE = Path.cwd() / 'analysis' / 'muon'
sys.path.insert(0, str(HERE))
SCRIPT = HERE / 't382_ral_silver_detector_share.py'
OUT = HERE / 'T382_ral_silver_detector_share'
SEED = 382
CALIBRATION = {'EMU00066572':20, 'EMU00066573':25, 'EMU00066574':20,
               'EMU00066575':25, 'EMU00066576':20, 'EMU00066577':25}
VALIDATION = {'EMU00066571':25, 'EMU00066584':20}
HOLDOUT = {'EMU00066578':63, 'EMU00066579':160, 'EMU00066580':400}
DIAGNOSTIC = {'EMU00066581':1000, 'EMU00066582':2000, 'EMU00066583':4000}


## Reproduce the frozen execution

The next cell deliberately runs the versioned module rather than duplicating
analysis code inside the notebook. It takes roughly 15–45 seconds with the
bundled Python runtime.


In [ ]:
import runpy
runpy.run_path(str(SCRIPT), run_name='__main__')


In [ ]:
result = json.loads((OUT / 'T382_DETECTOR_SHARE_RESULTS.json').read_text())
validation = json.loads((OUT / 'T382_DETECTOR_SHARE_VALIDATION.json').read_text())
runs = pd.read_csv(OUT / 'T382_DETECTOR_SHARE_RUNS.csv')
bins = pd.read_csv(OUT / 'T382_DETECTOR_SHARE_BIN_SENSITIVITY.csv')
result['status'], result['gates']


## Data-quality and construct checks

File hashes, native-bin checks, nonnegative/integer count checks and detector
coverage are in the validation JSON. The source is Class P: population
histograms. This supports population and detector-relation cuts but not an
individual-muon prediction claim.


In [ ]:
assert validation['all_data_quality_pass']
assert validation['individual_prediction_available'] is False
pd.DataFrame(validation['data_quality_by_run']).T


## ARA and physics views side by side

- ARA parent: `xP(t)=2(1-exp(-t/tau))`; its ridge is `tau*ln(2)`.
- ARA child: `xC(t)=1-cos(theta)`; the projected child is `xC/2`.
- Physics crosswalk: `theta=2*pi*gamma*B*t+phi0` from the detector-share phase relation.

The golden-ratio or any other irrationality coordinate is not inserted into
this test.


In [ ]:
runs[['run','split','field_g','detector_child_gain','reverse_gain',
      'free_gamma_mhz_per_g','phase_at_parent_ridge_rad']]


In [ ]:
bins.groupby('factor', as_index=False)['improvement'].mean()


## Takeaway

The parent cut is strong and reproducible. The candidate child cadence is
numerically close to the later-revealed muon reference, but the frozen spatial
relation does not generalize: it loses to the no-phase model on primary
holdouts and does not beat detector-map controls. That combination is a useful
source qualification result, not a neutrino-timing result.
